In [2]:
from gliner import GLiNER
from sklearn.metrics import classification_report

# -------------------------------
# Helper Functions
# -------------------------------

def read_conll_file(filepath):
    """
    Reads a 4-column CoNLL file and yields tokens and BIO labels per sentence.
    Columns: Token POS _ BIO-Label
    """
    tokens, labels = [], []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("-DOCSTART-"):
                if tokens:
                    yield tokens, labels
                    tokens, labels = [], []
                continue
            parts = line.split()
            if len(parts) < 4:
                continue
            tokens.append(parts[0])
            labels.append(parts[3])  # BIO label is in 4th column
        if tokens:
            yield tokens, labels

def bio_to_entities(tokens, bio_labels):
    """
    Convert BIO labels to entity spans with start/end indices.
    Returns a list of dicts: {"label": label, "tokens": [...], "start": i, "end": j}
    """
    entities = []
    entity = None
    for i, tag in enumerate(bio_labels):
        if tag.startswith("B-"):
            if entity:
                entities.append(entity)
            entity = {"label": tag[2:], "tokens": [tokens[i]], "start": i, "end": i}
        elif tag.startswith("I-") and entity and tag[2:] == entity["label"]:
            entity["tokens"].append(tokens[i])
            entity["end"] = i
        else:
            if entity:
                entities.append(entity)
                entity = None
    if entity:
        entities.append(entity)
    return entities

def entity_set_from_bio(tokens, bio_labels):
    """
    Convert BIO labels to a set of (text, label) tuples for evaluation.
    """
    ents = bio_to_entities(tokens, bio_labels)
    return set((" ".join(e["tokens"]), e["label"]) for e in ents)

def entity_set_from_pred(pred_entities):
    """
    Convert GLiNER predicted entities to a set of (text, label) tuples.
    """
    return set((e["text"], e["label"]) for e in pred_entities)

# -------------------------------
# Load GLiNER Model
# -------------------------------
model = GLiNER.from_pretrained("urchade/gliner_largev2")




Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

/opt/anaconda3/lib/python3.12/site-packages/transformers/convert_slow_tokenizer.py:561: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [4]:
# -------------------------------
# Evaluation Pipeline
# -------------------------------
conll_filepath = "advarsarial.conll"

# List of entities to evaluate
labels_list = [
    "EquipmentName",
    "DeviceProperties",
    "FaultCode",
    "FaultType",
    "MaintenanceMethod",
    "VehicleComponentLocation",
    "MeasurementValue",
    "VehicleMakeModel",
    "TimeDuration",
    "CustomerReportedSymptom"
]

all_true_labels = []
all_pred_labels = []

for tokens, labels in read_conll_file(conll_filepath):
    text = " ".join(tokens)
    
    # Get GLiNER predictions
    pred_entities = model.predict_entities(text, labels_list)
    
    # Convert BIO labels and predictions to sets
    true_ents = entity_set_from_bio(tokens, labels)
    pred_ents = entity_set_from_pred(pred_entities)
    
    # True positives and false negatives
    for ent in true_ents:
        if ent in pred_ents:
            all_true_labels.append(ent[1])
            all_pred_labels.append(ent[1])
        else:
            all_true_labels.append(ent[1])
            all_pred_labels.append("O")  # False negative

    # False positives
    for ent in pred_ents:
        if ent not in true_ents:
            all_true_labels.append("O")
            all_pred_labels.append(ent[1])

# -------------------------------
# Generate Classification Report
# -------------------------------
report = classification_report(
    all_true_labels,
    all_pred_labels,
    labels=labels_list,
    zero_division=0
)

print(report)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


                          precision    recall  f1-score   support

           EquipmentName       0.24      0.15      0.19       288
        DeviceProperties       0.00      0.00      0.00       250
               FaultCode       0.03      0.05      0.04       349
               FaultType       0.25      0.56      0.35       540
       MaintenanceMethod       0.59      0.71      0.64       247
VehicleComponentLocation       0.28      0.99      0.44       282
        MeasurementValue       0.52      1.00      0.68       304
        VehicleMakeModel       0.68      1.00      0.81       535
            TimeDuration       0.63      0.76      0.69       310
 CustomerReportedSymptom       0.00      0.00      0.00       479

               micro avg       0.38      0.53      0.44      3584
               macro avg       0.32      0.52      0.38      3584
            weighted avg       0.32      0.53      0.39      3584



In [6]:
# -------------------------------
# Evaluation Pipeline
# -------------------------------
conll_filepath = "goldset.conll"

# List of entities to evaluate
labels_list = [
    "EquipmentName",
    "DeviceProperties",
    "FaultCode",
    "FaultType",
    "MaintenanceMethod",
    "VehicleComponentLocation",
    "MeasurementValue",
    "VehicleMakeModel",
    "TimeDuration",
    "CustomerReportedSymptom"
]

all_true_labels = []
all_pred_labels = []

for tokens, labels in read_conll_file(conll_filepath):
    text = " ".join(tokens)
    
    # Get GLiNER predictions
    pred_entities = model.predict_entities(text, labels_list)
    
    # Convert BIO labels and predictions to sets
    true_ents = entity_set_from_bio(tokens, labels)
    pred_ents = entity_set_from_pred(pred_entities)
    
    # True positives and false negatives
    for ent in true_ents:
        if ent in pred_ents:
            all_true_labels.append(ent[1])
            all_pred_labels.append(ent[1])
        else:
            all_true_labels.append(ent[1])
            all_pred_labels.append("O")  # False negative

    # False positives
    for ent in pred_ents:
        if ent not in true_ents:
            all_true_labels.append("O")
            all_pred_labels.append(ent[1])

# -------------------------------
# Generate Classification Report
# -------------------------------
report = classification_report(
    all_true_labels,
    all_pred_labels,
    labels=labels_list,
    zero_division=0
)

print(report)

                          precision    recall  f1-score   support

           EquipmentName       0.57      0.10      0.17       121
        DeviceProperties       0.00      0.00      0.00        32
               FaultCode       0.15      0.33      0.21        18
               FaultType       0.24      0.32      0.27        22
       MaintenanceMethod       0.17      0.06      0.09        69
VehicleComponentLocation       0.07      0.22      0.11        18
        MeasurementValue       0.43      0.48      0.46        27
        VehicleMakeModel       0.00      0.00      0.00         2
            TimeDuration       0.17      0.12      0.14         8
 CustomerReportedSymptom       0.00      0.00      0.00        40

               micro avg       0.20      0.13      0.16       357
               macro avg       0.18      0.16      0.14       357
            weighted avg       0.29      0.13      0.14       357

